In [24]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.appName("JoinStrategies")
    .config("spark.executor.memory", "1g")
    # .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

# Dataset 1: Users (5 rows)
users_data = [
    (1, "Alice", "Germany"),
    (2, "Bob", "USA"),
    (3, "Charlie", "UK"),
    (4, "Diana", "France"),
    (5, "Ethan", "Canada"),  # Ethan has no orders yet
]

users_df = spark.createDataFrame(
    users_data, schema=["user_id", "name", "country"]
)

# Dataset 2: Orders (8 rows)
orders_data = [
    (101, 1, "Laptop", 1200.0),
    (102, 2, "Desk Chair", 250.0),
    (103, 1, "Wireless Mouse", 35.0),
    (104, 3, "Standing Desk", 550.0),
    (105, 2, "Monitor", 300.0),
    (106, 4, "Pen Set", 15.0),
    (107, 1, "USB-C Hub", 60.0),
    (108, 3, "Desk Lamp", 45.0),
]

orders_df = spark.createDataFrame(
    orders_data, schema=["order_id", "user_id", "product", "amount"]
)

In [2]:
orders_df.show(20)

+--------+-------+--------------+------+
|order_id|user_id|       product|amount|
+--------+-------+--------------+------+
|     101|      1|        Laptop|1200.0|
|     102|      2|    Desk Chair| 250.0|
|     103|      1|Wireless Mouse|  35.0|
|     104|      3| Standing Desk| 550.0|
|     105|      2|       Monitor| 300.0|
|     106|      4|       Pen Set|  15.0|
|     107|      1|     USB-C Hub|  60.0|
|     108|      3|     Desk Lamp|  45.0|
+--------+-------+--------------+------+



In [3]:
users_df.show(20)

+-------+-------+-------+
|user_id|   name|country|
+-------+-------+-------+
|      1|  Alice|Germany|
|      2|    Bob|    USA|
|      3|Charlie|     UK|
|      4|  Diana| France|
|      5|  Ethan| Canada|
+-------+-------+-------+



### SORT MERGE JOIN - SMJ

In [21]:
# Step 1: Join orders with users on the shared 'user_id' key
joined_df = orders_df.join(users_df, on="user_id", how="inner")

# Step 2: Select clean, readable columns
final_join_df = joined_df.select(
    "order_id", "user_id", "name", "country", "product", "amount"
)

final_join_df.explain(mode="extended")

== Parsed Logical Plan ==
'Project ['order_id, 'user_id, 'name, 'country, 'product, 'amount]
+- Project [user_id#138L, order_id#137L, product#139, amount#140, name#135, country#136]
   +- Join Inner, (user_id#138L = user_id#134L)
      :- LogicalRDD [order_id#137L, user_id#138L, product#139, amount#140], false
      +- LogicalRDD [user_id#134L, name#135, country#136], false

== Analyzed Logical Plan ==
order_id: bigint, user_id: bigint, name: string, country: string, product: string, amount: double
Project [order_id#137L, user_id#138L, name#135, country#136, product#139, amount#140]
+- Project [user_id#138L, order_id#137L, product#139, amount#140, name#135, country#136]
   +- Join Inner, (user_id#138L = user_id#134L)
      :- LogicalRDD [order_id#137L, user_id#138L, product#139, amount#140], false
      +- LogicalRDD [user_id#134L, name#135, country#136], false

== Optimized Logical Plan ==
Project [order_id#137L, user_id#138L, name#135, country#136, product#139, amount#140]
+- Join In

In [22]:
final_join_df.show()

[Stage 1:=============================>                             (1 + 1) / 2]

+--------+-------+-------+-------+--------------+------+
|order_id|user_id|   name|country|       product|amount|
+--------+-------+-------+-------+--------------+------+
|     101|      1|  Alice|Germany|        Laptop|1200.0|
|     103|      1|  Alice|Germany|Wireless Mouse|  35.0|
|     107|      1|  Alice|Germany|     USB-C Hub|  60.0|
|     102|      2|    Bob|    USA|    Desk Chair| 250.0|
|     105|      2|    Bob|    USA|       Monitor| 300.0|
|     104|      3|Charlie|     UK| Standing Desk| 550.0|
|     108|      3|Charlie|     UK|     Desk Lamp|  45.0|
|     106|      4|  Diana| France|       Pen Set|  15.0|
+--------+-------+-------+-------+--------------+------+



### BROADCAST JOIN

In [25]:

broadcast_join_df = orders_df.join(F.broadcast(users_df), on="user_id", how="inner")

final_bhj_df = broadcast_join_df.select(
    "order_id", "user_id", "name", "country", "product", "amount"
)

final_bhj_df.explain(mode="extended")

== Parsed Logical Plan ==
'Project ['order_id, 'user_id, 'name, 'country, 'product, 'amount]
+- Project [user_id#164L, order_id#163L, product#165, amount#166, name#161, country#162]
   +- Join Inner, (user_id#164L = user_id#160L)
      :- LogicalRDD [order_id#163L, user_id#164L, product#165, amount#166], false
      +- ResolvedHint (strategy=broadcast)
         +- LogicalRDD [user_id#160L, name#161, country#162], false

== Analyzed Logical Plan ==
order_id: bigint, user_id: bigint, name: string, country: string, product: string, amount: double
Project [order_id#163L, user_id#164L, name#161, country#162, product#165, amount#166]
+- Project [user_id#164L, order_id#163L, product#165, amount#166, name#161, country#162]
   +- Join Inner, (user_id#164L = user_id#160L)
      :- LogicalRDD [order_id#163L, user_id#164L, product#165, amount#166], false
      +- ResolvedHint (strategy=broadcast)
         +- LogicalRDD [user_id#160L, name#161, country#162], false

== Optimized Logical Plan ==
Proj

In [26]:
final_bhj_df.show()

+--------+-------+-------+-------+--------------+------+
|order_id|user_id|   name|country|       product|amount|
+--------+-------+-------+-------+--------------+------+
|     101|      1|  Alice|Germany|        Laptop|1200.0|
|     102|      2|    Bob|    USA|    Desk Chair| 250.0|
|     103|      1|  Alice|Germany|Wireless Mouse|  35.0|
|     104|      3|Charlie|     UK| Standing Desk| 550.0|
|     105|      2|    Bob|    USA|       Monitor| 300.0|
|     106|      4|  Diana| France|       Pen Set|  15.0|
|     107|      1|  Alice|Germany|     USB-C Hub|  60.0|
|     108|      3|Charlie|     UK|     Desk Lamp|  45.0|
+--------+-------+-------+-------+--------------+------+



In [27]:
spark.stop()